<a href="https://colab.research.google.com/github/danielcosta04/LinguagemProgramacao/blob/main/Aula%203%3A%20Limpeza%2C%20Prepara%C3%A7%C3%A3o%20e%20Qualidade%20dos%20Dados%20com%20Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas**

Como usar:

Abra o Google Colab e crie um novo notebook
Faça upload do arquivo vendas_brasil_aula3_bruto.csv
Cole as células abaixo na ordem

**Atividade Prática Aula 3**

In [ ]:
# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas
**Arquivo:** `vendas_brasil_aula3_bruto.csv`

In [ ]:
import pandas as pd
import numpy as np

# Configuração para melhor visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

**2. Leitura da base bruta**

In [ ]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')
print("Shape inicial:", df.shape)
df.head()

In [ ]:
df.tail()

**3. Check-up inicial do dataset**

In [ ]:
print("=== CHECK-UP INICIAL ===")
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}\n")

print("Tipos de dados:")
print(df.dtypes)

print("\nValores nulos por coluna:")
print(df.isna().sum())

print("\nExemplo de valores problemáticos:")
print(df[['data', 'receita', 'lucro', 'quantidade']].head(15))

**4. Batalha 1 — Conversão de tipos**

In [ ]:
# Copia para trabalhar com segurança
df_clean = df.copy()

# 1. Data
df_clean['data'] = pd.to_datetime(df_clean['data'], errors='coerce')

# 2. Quantidade
df_clean['quantidade'] = pd.to_numeric(df_clean['quantidade'], errors='coerce')

# 3. Receita e Lucro (muitos valores com texto, "zero", "R$", espaços, etc.)
def limpar_valor_monetario(coluna):
    return (
        coluna.astype(str)
        .str.replace('R$', '', regex=False)
        .str.replace('.', '', regex=False)      # remove ponto de milhar
        .str.replace(',', '.', regex=False)     # troca vírgula decimal
        .str.replace('zero', '0', regex=False)
        .str.strip()
    )

df_clean['receita'] = pd.to_numeric(limpar_valor_monetario(df_clean['receita']), errors='coerce')
df_clean['lucro']  = pd.to_numeric(limpar_valor_monetario(df_clean['lucro']),  errors='coerce')

print("Tipos após conversão:")
print(df_clean[['data', 'quantidade', 'receita', 'lucro']].dtypes)
print("\nDatas inválidas:", df_clean['data'].isna().sum())

**Reflexão:**

Deixar data como texto impede cálculos temporais (mês, trimestre, evolução).
Deixar receita como string impede somas, médias e KPIs.

**5. Batalha 2 — Tratamento de valores ausentes**

In [ ]:
print("Nulos antes do tratamento:")
print(df_clean.isna().sum())

# Estratégia adotada:
# - data inválida → remover (não dá para analisar sem data)
# - receita e lucro nulos → preencher com 0 quando faz sentido (venda sem valor informado)
# - uf e canal nulos → preencher com "Não informado"
# - observacao → manter nulo (não é crítica)

df_clean = df_clean.dropna(subset=['data'])                    # remove datas inválidas
df_clean['receita'] = df_clean['receita'].fillna(0)
df_clean['lucro']   = df_clean['lucro'].fillna(0)
df_clean['uf']      = df_clean['uf'].fillna('Não informado')
df_clean['canal']   = df_clean['canal'].fillna('Não informado')
df_clean['categoria'] = df_clean['categoria'].fillna('Não informado')

print("\nNulos após tratamento:")
print(df_clean.isna().sum())

**Justificativa:**

Removi linhas sem data porque é impossível fazer análise temporal.
Preenchi receita/lucro com 0 quando ausente, pois são campos numéricos críticos para KPI.
"Não informado" em uf/canal permite manter o registro sem distorcer agrupamentos.

**6. Batalha 3 — Duplicidades**

In [ ]:
# Verificar duplicatas totais
print("Duplicatas totais:", df_clean.duplicated().sum())

# Verificar duplicatas baseadas em pedido_id (chave principal)
print("Duplicatas por pedido_id:", df_clean.duplicated(subset=['pedido_id']).sum())

# Remover duplicatas completas (mantendo a primeira ocorrência)
df_clean = df_clean.drop_duplicates()

print("Shape após remoção de duplicatas:", df_clean.shape)

**7. Padronização de categorias**

In [ ]:
# UF
df_clean['uf'] = df_clean['uf'].astype(str).str.strip().str.upper()
df_clean['uf'] = df_clean['uf'].replace({'RJ': 'RJ', 'SP': 'SP', 'MG': 'MG',
                                         'PR': 'PR', 'RS': 'RS', 'SC': 'SC',
                                         'BA': 'BA', 'ES': 'ES', 'Não informado': 'Não informado'})

# Canal
df_clean['canal'] = df_clean['canal'].astype(str).str.strip().str.title()
df_clean['canal'] = df_clean['canal'].replace({
    'Online': 'Online',
    'Marketplace': 'Marketplace',
    'Loja Física': 'Loja Física',
    'Loja Fisica': 'Loja Física',
    'ONLINE': 'Online',
    'Marketplace ': 'Marketplace',
    'Não informado': 'Não informado'
})

# Categoria
df_clean['categoria'] = df_clean['categoria'].astype(str).str.strip().str.title()
df_clean['categoria'] = df_clean['categoria'].replace({
    'Telefonia ': 'Telefonia',
    'Perifericos': 'Periféricos',
    'telefonia ': 'Telefonia',
    'Não informado': 'Não informado'
})

print("UF únicos:", sorted(df_clean['uf'].unique()))
print("Canal únicos:", df_clean['canal'].unique())
print("Categoria únicos:", df_clean['categoria'].unique())

**8. Variáveis derivadas**

In [ ]:
# Variáveis temporais
df_clean['ano'] = df_clean['data'].dt.year
df_clean['mes'] = df_clean['data'].dt.month
df_clean['ano_mes'] = df_clean['data'].dt.to_period('M')

# Margem de lucro (com proteção contra divisão por zero)
df_clean['margem_lucro'] = np.where(
    df_clean['receita'] > 0,
    (df_clean['lucro'] / df_clean['receita']) * 100,
    0
)

# Remover possíveis infinitos
df_clean['margem_lucro'] = df_clean['margem_lucro'].replace([np.inf, -np.inf], 0)

df_clean[['data', 'ano', 'mes', 'ano_mes', 'receita', 'lucro', 'margem_lucro']].head()

**Reflexão:**
Quando receita = 0, margem_lucro ficaria infinita ou NaN. Decidi colocar 0% nesses casos para não quebrar análises.

**9. Seleção final da base clean**

In [ ]:
# Colunas finais para dashboard/análise
cols_finais = ['pedido_id', 'data', 'ano', 'mes', 'ano_mes', 'uf', 'canal',
               'categoria', 'produto', 'quantidade', 'receita', 'lucro', 'margem_lucro', 'observacao']

df_final = df_clean[cols_finais].copy()
print("Shape da base final:", df_final.shape)
df_final.head()

**10. Validação final**

In [ ]:
print("=== VALIDAÇÃO FINAL ===")
print("Tipos:\n", df_final.dtypes)
print("\nNulos restantes:\n", df_final.isna().sum())
print("\nDuplicatas restantes:", df_final.duplicated().sum())
print("\nInfinitos em margem_lucro:", np.isinf(df_final['margem_lucro']).sum())

**11. Exportação da base limpa**

In [ ]:
df_final.to_csv('vendas_brasil_aula3_clean.csv', index=False)
print("✅ Base limpa exportada com sucesso: vendas_brasil_aula3_clean.csv")

 **12. Conclusão e registro reflexivo
Principais problemas encontrados:**

Datas em formatos diferentes e inválidas (data_invalida)
Valores monetários com "R$", "zero", espaços e vírgulas
Variações de escrita em UF, canal e categoria
Duplicatas (algumas linhas repetidas)
Muitos valores nulos em campos críticos

Decisão mais difícil:
Decidir o que fazer com linhas onde receita = 0 ou lucro estava ausente. Optei por manter os registros preenchendo com 0, pois remoção em massa perderia volume importante de vendas.
Impacto em um dashboard:
Sem limpeza, gráficos de receita ficariam errados, filtros por UF/canal quebrariam, evolução temporal seria impossível e KPIs (como margem de lucro) dariam resultados absurdos.
Por que limpeza é a "fundação invisível"?
Porque quando o dashboard está bonito e funcionando, ninguém vê o trabalho sujo que foi feito antes. Mas se a limpeza for mal feita, todo o dashboard perde credibilidade.

**13. Desafio Extra (bônus)**

In [ ]:
# Desafio extra
print("UF com maior receita:")
print(df_final.groupby('uf')['receita'].sum().sort_values(ascending=False).head())

print("\nCanal com maior lucro:")
print(df_final.groupby('canal')['lucro'].sum().sort_values(ascending=False))

print("\nReceita por mês:")
print(df_final.groupby('ano_mes')['receita'].sum().sort_values(ascending=False).head(6))